In [3]:
# Notebook: Q1_Extra_Checks_v01.ipynb | Cell 1 (v2)
# (A) Observability-mask threshold sensitivity: tree < 0.2 / 0.3 / 0.4 (mangrove <= 0.1 fixed).
# v2 adds: hidden-population poverty + unobservable share, common-test-set ROC (Tree < 0.2 rows), pre-set verdict.
from pathlib import Path
import json
import numpy as np
import pandas as pd
from lightgbm import LGBMClassifier
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import roc_auc_score, average_precision_score
from scipy.stats import spearmanr

# ---- Settings ----
ROOT = Path(r"D:\Siam\Thesis_GeoAI")
LADDER_DIR = ROOT / "05_models" / "q1_ladder_v01"
MASTER = ROOT / "04_master_table" / "master_v06_20261020.csv"
HAZ_V02 = ROOT / "05_models" / "final_v02" / "hazard_predictions_v02.csv"
OUT_DIR = ROOT / "05_models" / "q1_extra_v01"
FEAT_V03 = ["Elevation_Mean_m", "Slope_Mean_deg", "Distance_to_River_m", "Cropland_Fraction"]
THRESHOLDS = [0.2, 0.3, 0.4]
MANGROVE_MAX = 0.1
COMMON_TREE_MAX = 0.2           # common test set = rows observable under the strictest mask
ROC_TOL = 0.03                  # pre-set rule: |ROC(thr) - ROC(0.3)| <= 0.03
SEED, N_FOLDS = 42, 5
REF_ROC, REF_HIDDEN_POP = 0.702, 319_900       # published values at threshold 0.3

OUT_DIR.mkdir(parents=True, exist_ok=True)
d = pd.read_parquet(LADDER_DIR / "ladder_rows_v01.parquet")
params = json.loads((LADDER_DIR / "ladder_params_v01.json").read_text())
params.update({"random_state": SEED, "verbose": -1})
print("LGBM params:", params)

m = pd.read_csv(MASTER, usecols=["Grid_ID", "Is_Water", "Population_Count", "Poverty_Rate_Upazila_Pct",
                                 "Tree_Fraction", "Mangrove_Fraction_GMW"] + FEAT_V03)
m = m[m["Is_Water"] == 0].reset_index(drop=True)
assert len(m) == 160874 and m["Grid_ID"].is_unique
print("NaN counts (model cells):", m[["Tree_Fraction", "Mangrove_Fraction_GMW", "Poverty_Rate_Upazila_Pct"]]
      .isna().sum().to_dict())
m["Mangrove_Fraction_GMW"] = m["Mangrove_Fraction_GMW"].fillna(0)
m["Population_Count"] = m["Population_Count"].fillna(0)

d = d.merge(m[["Grid_ID", "Mangrove_Fraction_GMW"]], on="Grid_ID", how="left")
assert d["Mangrove_Fraction_GMW"].notna().all() and d["Tree_Fraction"].notna().all()

h2 = pd.read_csv(HAZ_V02, usecols=["Grid_ID", "Hazard_Prob_LGBM"])
m = m.merge(h2.rename(columns={"Hazard_Prob_LGBM": "P_v02"}), on="Grid_ID", how="left")
assert m["P_v02"].notna().all()

def pct(x):
    x = pd.Series(np.asarray(x))
    return ((x.rank(method="average") - 0.5) / len(x)).to_numpy()

def wpov(mask):
    s = m.loc[np.asarray(mask) & m["Poverty_Rate_Upazila_Pct"].notna().to_numpy()]
    return float((s["Population_Count"] * s["Poverty_Rate_Upazila_Pct"]).sum() / s["Population_Count"].sum()) \
        if s["Population_Count"].sum() > 0 else np.nan

m["Pct_v02"] = pct(m["P_v02"])
total_pos = int(d["Flooded_Event_Raw_v01"].sum())
pop_total = m["Population_Count"].sum()
pov_all = wpov(np.ones(len(m), dtype=bool))
print(f"Pop-weighted poverty, all model cells: {pov_all:.1f}% (published 23.2%)")

rows, pct_by_thr = [], {}
for thr in THRESHOLDS:
    obs = (d["Tree_Fraction"] < thr) & (d["Mangrove_Fraction_GMW"] <= MANGROVE_MAX)
    dd = d[obs].reset_index(drop=True)
    y = dd["Flooded_Event_Raw_v01"].astype(int).to_numpy()
    oof = np.full(len(dd), np.nan)
    cv = StratifiedGroupKFold(N_FOLDS, shuffle=True, random_state=SEED)
    for tr, te in cv.split(dd[FEAT_V03], y, dd["Block_ID"]):
        mdl = LGBMClassifier(**params).fit(dd[FEAT_V03].iloc[tr], y[tr])
        oof[te] = mdl.predict_proba(dd[FEAT_V03].iloc[te])[:, 1]
    assert not np.isnan(oof).any()
    cs = (dd["Tree_Fraction"] < COMMON_TREE_MAX).to_numpy()
    final = LGBMClassifier(**params).fit(dd[FEAT_V03], y)
    P = pct(final.predict_proba(m[FEAT_V03])[:, 1])
    pct_by_thr[thr] = P
    hidden = (m["Pct_v02"].to_numpy() < 0.2) & (P >= 0.6)
    m_obs = ((m["Tree_Fraction"] < thr) & (m["Mangrove_Fraction_GMW"] <= MANGROVE_MAX)).to_numpy()
    hid_pop = float(m.loc[hidden, "Population_Count"].sum())
    rows.append({"Tree_Threshold": thr, "N_Rows": len(dd), "N_Cells": dd["Grid_ID"].nunique(),
                 "Positives_Kept_Pct": 100 * y.sum() / total_pos, "Prevalence": y.mean(),
                 "Observable_Pop_Pct": 100 * m.loc[m_obs, "Population_Count"].sum() / pop_total,
                 "ROC_AUC": roc_auc_score(y, oof), "PR_AUC": average_precision_score(y, oof),
                 "ROC_AUC_Common_Tree02": roc_auc_score(y[cs], oof[cs]),
                 "Hidden_Cells": int(hidden.sum()), "Hidden_Pop": hid_pop,
                 "Hidden_Unobs_Pop_Pct": 100 * m.loc[hidden & ~m_obs, "Population_Count"].sum() / hid_pop
                                         if hid_pop > 0 else np.nan,
                 "Hidden_Poverty_Pct": wpov(hidden)})
    r = rows[-1]
    print(f"thr {thr}: ROC {r['ROC_AUC']:.3f} | common-set ROC {r['ROC_AUC_Common_Tree02']:.3f} | "
          f"hidden pop {hid_pop:,.0f} | hidden poverty {r['Hidden_Poverty_Pct']:.1f}%")

res = pd.DataFrame(rows)
ref = pct_by_thr[0.3]
top_ref = ref >= np.quantile(ref, 0.9)
res["Spearman_vs_0.3"] = [spearmanr(pct_by_thr[t], ref).correlation for t in THRESHOLDS]
res["Top10_Overlap_vs_0.3_Pct"] = [100 * ((pct_by_thr[t] >= np.quantile(pct_by_thr[t], 0.9)) & top_ref).sum()
                                   / top_ref.sum() for t in THRESHOLDS]
pd.set_option("display.width", 250)
print("\n", res.round(3).to_string(index=False))

r03 = res.set_index("Tree_Threshold").loc[0.3]
print(f"\nReproduction @0.3: ROC {r03['ROC_AUC']:.3f} vs {REF_ROC} -> "
      f"{'OK' if abs(r03['ROC_AUC'] - REF_ROC) <= 0.01 else 'DIFFERENT'} | hidden pop {r03['Hidden_Pop']:,.0f} "
      f"vs {REF_HIDDEN_POP:,} -> {'OK' if abs(r03['Hidden_Pop'] / REF_HIDDEN_POP - 1) <= 0.05 else 'DIFFERENT'}")

print("\n=== Pre-set rule (fixed before running) ===")
all_ok = True
for thr in [0.2, 0.4]:
    r = res.set_index("Tree_Threshold").loc[thr]
    ok_roc = abs(r["ROC_AUC"] - r03["ROC_AUC"]) <= ROC_TOL
    ok_dir = (r["Hidden_Pop"] > 0) and (r["Hidden_Poverty_Pct"] > pov_all)
    all_ok &= ok_roc and ok_dir
    print(f"thr {thr}: dROC {r['ROC_AUC'] - r03['ROC_AUC']:+.3f} -> {'PASS' if ok_roc else 'FAIL'} | "
          f"hidden pop > 0 and poorer than average -> {'PASS' if ok_dir else 'FAIL'}")
print("VERDICT:", "ROBUST to mask threshold" if all_ok else "NOT ROBUST (report honestly)")

res.to_csv(OUT_DIR / "mask_threshold_sensitivity_v01.csv", index=False)
print("Saved:", OUT_DIR / "mask_threshold_sensitivity_v01.csv")

LGBM params: {'n_estimators': 500, 'learning_rate': 0.03, 'num_leaves': 31, 'min_child_samples': 50, 'subsample': 0.8, 'subsample_freq': 1, 'random_state': 42, 'n_jobs': -1, 'verbose': -1}
NaN counts (model cells): {'Tree_Fraction': 0, 'Mangrove_Fraction_GMW': 0, 'Poverty_Rate_Upazila_Pct': 0}
Pop-weighted poverty, all model cells: 23.2% (published 23.2%)
thr 0.2: ROC 0.668 | common-set ROC 0.668 | hidden pop 293,790 | hidden poverty 26.8%
thr 0.3: ROC 0.702 | common-set ROC 0.677 | hidden pop 319,905 | hidden poverty 27.1%
thr 0.4: ROC 0.747 | common-set ROC 0.684 | hidden pop 385,826 | hidden poverty 26.8%

  Tree_Threshold  N_Rows  N_Cells  Positives_Kept_Pct  Prevalence  Observable_Pop_Pct  ROC_AUC  PR_AUC  ROC_AUC_Common_Tree02  Hidden_Cells  Hidden_Pop  Hidden_Unobs_Pop_Pct  Hidden_Poverty_Pct  Spearman_vs_0.3  Top10_Overlap_vs_0.3_Pct
            0.2  118002    39334              89.054       0.155              13.100    0.668   0.255                  0.668          3813  293790

In [4]:
# Notebook: Q1_Extra_Checks_v01.ipynb | Cell 2
# (B) Typical published protocol (50:50 balancing, random 80/20 split, accuracy/F1) vs spatial split vs honest set-up.
from pathlib import Path
import json
import numpy as np
import pandas as pd
from lightgbm import LGBMClassifier
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

# ---- Settings ----
ROOT = Path(r"D:\Siam\Thesis_GeoAI")
LADDER_DIR = ROOT / "05_models" / "q1_ladder_v01"
OUT_DIR = ROOT / "05_models" / "q1_extra_v01"
FEAT_V02 = ["Elevation_Mean_m", "Slope_Mean_deg", "Distance_to_River_m", "Tree_Fraction", "NDVI_Mean"]
FEAT_V03 = ["Elevation_Mean_m", "Slope_Mean_deg", "Distance_to_River_m", "Cropland_Fraction"]
SEED, TEST_SIZE = 42, 0.2

d = pd.read_parquet(LADDER_DIR / "ladder_rows_v01.parquet")
params = json.loads((LADDER_DIR / "ladder_params_v01.json").read_text())
params.update({"random_state": SEED, "verbose": -1})

def balanced(df, target):
    pos = df[df[target] == 1]
    neg = df[df[target] == 0].sample(n=len(pos), random_state=SEED)
    return pd.concat([pos, neg]).sample(frac=1, random_state=SEED).reset_index(drop=True)

def run(name, df, target, feats, split):
    y = df[target].astype(int).to_numpy()
    if split == "random":
        tr, te = train_test_split(np.arange(len(df)), test_size=TEST_SIZE, stratify=y, random_state=SEED)
    else:
        tr, te = next(GroupShuffleSplit(1, test_size=TEST_SIZE, random_state=SEED).split(df, y, df["Block_ID"]))
    mdl = LGBMClassifier(**params).fit(df[feats].iloc[tr], y[tr])
    p = mdl.predict_proba(df[feats].iloc[te])[:, 1]
    yt = y[te]
    r = {"Setup": name, "Test_Rows": len(te), "Test_Prevalence": yt.mean(),
         "Accuracy": accuracy_score(yt, p >= 0.5), "F1": f1_score(yt, p >= 0.5), "ROC_AUC": roc_auc_score(yt, p)}
    obs = df["Is_Observable_Row"].to_numpy()[te] == 1
    r["ROC_AUC_Observable_Test_Only"] = roc_auc_score(yt[obs], p[obs]) if 0 < yt[obs].sum() < obs.sum() else np.nan
    return r

naive_bal = balanced(d, "Flooded_Event_Raw_v01")
honest_bal = balanced(d[d["Is_Observable_Row"] == 1], "Flooded_Event")
print("Balanced rows: naive", len(naive_bal), "| honest", len(honest_bal))

res = pd.DataFrame([
    run("K1 naive labels, balanced, random 80/20", naive_bal, "Flooded_Event_Raw_v01", FEAT_V02, "random"),
    run("K2 naive labels, balanced, 15 km block 80/20", naive_bal, "Flooded_Event_Raw_v01", FEAT_V02, "block"),
    run("K3 observable labels, balanced, 15 km block 80/20", honest_bal, "Flooded_Event", FEAT_V03, "block"),
])
pd.set_option("display.width", 220)
print(res.round(3).to_string(index=False))
res.to_csv(OUT_DIR / "kuet_style_protocol_v01.csv", index=False)
print("Saved:", OUT_DIR / "kuet_style_protocol_v01.csv")

Balanced rows: naive 41036 | honest 39764
                                            Setup  Test_Rows  Test_Prevalence  Accuracy    F1  ROC_AUC  ROC_AUC_Observable_Test_Only
          K1 naive labels, balanced, random 80/20       8208            0.500     0.864 0.874    0.922                         0.744
     K2 naive labels, balanced, 15 km block 80/20       7444            0.475     0.871 0.871    0.928                         0.692
K3 observable labels, balanced, 15 km block 80/20       9167            0.519     0.688 0.708    0.742                         0.742
Saved: D:\Siam\Thesis_GeoAI\05_models\q1_extra_v01\kuet_style_protocol_v01.csv


In [5]:
# Notebook: Q1_Extra_Checks_v01.ipynb | Cell 3 (descriptive only; run after Cell 1 in the same kernel)
# Access (road / hospital distance) of hidden-hazard cells vs all cells. No model, no new rule.
import numpy as np
import pandas as pd

ACC = ["Distance_to_Road_m", "Distance_to_Hospital_m"]
acc = pd.read_csv(MASTER, usecols=["Grid_ID"] + ACC)

mm = m[["Grid_ID", "Population_Count", "Pct_v02"]].copy()
mm["Pct_v03"] = pct_by_thr[0.3]                     # same order as m (from Cell 1)
mm = mm.merge(acc, on="Grid_ID", how="left")
assert len(mm) == 160874 and mm["Grid_ID"].is_unique
print("NaN in access columns:", mm[ACC].isna().sum().to_dict())

mm["Hidden"] = (mm["Pct_v02"] < 0.2) & (mm["Pct_v03"] >= 0.6)
print(f"Hidden cells {int(mm['Hidden'].sum())} (expected 4,124) | "
      f"hidden pop {mm.loc[mm['Hidden'], 'Population_Count'].sum():,.0f} (expected 319,905)")
assert int(mm["Hidden"].sum()) == 4124

def wmedian(v, w):
    o = np.argsort(v)
    v, w = v[o], w[o]
    c = np.cumsum(w)
    return float(v[np.searchsorted(c, c[-1] / 2)])

groups = {"Hidden hazard": mm["Hidden"].to_numpy(),
          "All other cells": ~mm["Hidden"].to_numpy(),
          "All cells": np.ones(len(mm), dtype=bool)}
rows = []
for name, g in groups.items():
    s = mm[g & mm[ACC].notna().all(axis=1).to_numpy() & (mm["Population_Count"] > 0).to_numpy()]
    w = s["Population_Count"].to_numpy()
    r = {"Group": name, "Cells_With_Pop": len(s), "Pop": float(w.sum())}
    for c in ACC:
        v = s[c].to_numpy() / 1000.0
        k = c.replace("_m", "_km")
        r[k + "_PopW_Mean"] = np.average(v, weights=w)
        r[k + "_PopW_Median"] = wmedian(v, w)
    rows.append(r)

res3 = pd.DataFrame(rows)
pd.set_option("display.width", 250)
print("\n", res3.round(2).to_string(index=False))

h = res3.set_index("Group")
worse = all(h.loc["Hidden hazard", c.replace("_m", "_km") + "_PopW_Median"]
            > h.loc["All cells", c.replace("_m", "_km") + "_PopW_Median"] for c in ACC)
print("\nPre-set rule (both medians higher than all cells):",
      "WORSE ACCESS" if worse else "ACCESS SIMILAR / MIXED")
res3.to_csv(OUT_DIR / "hidden_hazard_access_v01.csv", index=False)
print("Saved:", OUT_DIR / "hidden_hazard_access_v01.csv")

NaN in access columns: {'Distance_to_Road_m': 0, 'Distance_to_Hospital_m': 0}
Hidden cells 4124 (expected 4,124) | hidden pop 319,905 (expected 319,905)

           Group  Cells_With_Pop        Pop  Distance_to_Road_km_PopW_Mean  Distance_to_Road_km_PopW_Median  Distance_to_Hospital_km_PopW_Mean  Distance_to_Hospital_km_PopW_Median
  Hidden hazard            4049  319905.37                           0.55                             0.19                               5.90                                 5.69
All other cells          120299 7405210.38                           0.37                             0.13                               5.63                                 4.96
      All cells          124348 7725115.75                           0.38                             0.13                               5.64                                 4.99

Pre-set rule (both medians higher than all cells): WORSE ACCESS
Saved: D:\Siam\Thesis_GeoAI\05_models\q1_extra_v01\hidden_hazard